# 03 · Modelos de línea base (Etapa II)

**Fase CRISP-DM:** modelado. Se usan los modelos de línea base que
propone el informe:

- **Clase mayoritaria:** referencia mínima.
- **Regresión logística:** modelo lineal con log y estandarización.
- **Árbol de decisión:** modelo no lineal e interpretable.

Experimentos:

1. **Binario V1** (benigno vs. DDoS), entrenado solo con benigno + SYN y
   evaluado con todas las clases. Mide qué ocurre ante una variante de
   DDoS no vista (UDP).
2. **Binario V2**, entrenado con benigno + SYN + UDP.
3. **Multiclase:** benigno, DDoS SYN y DDoS UDP.

Todos los modelos se evalúan en validación y en la prueba oficial
(dentro del dominio IoMT). Las métricas se guardan en
`deliverables/resultados/`.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ.resolve()))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

from src.data_loader import CLASES, CARPETA_MODELOS, cargar_procesado, cargar_variables
from src.features import preprocesador
from src.evaluate import evaluar, matriz_confusion, guardar_tabla

config = cargar_variables()
variables = config["variables"]
SEMILLA = config["semilla"]

train = cargar_procesado("train")
val = cargar_procesado("val")
test = cargar_procesado("test")

print("Variables:", len(variables))
print("Entrenamiento:", train.shape, "| Validación:", val.shape, "| Prueba oficial:", test.shape)

In [ ]:
def modelos_base():
    """Modelos nuevos sin entrenar; se crean de nuevo en cada experimento."""
    return {
        "Clase mayoritaria": make_pipeline(
            preprocesador(),
            DummyClassifier(strategy="most_frequent"),
        ),
        "Regresión logística": make_pipeline(
            preprocesador(escalar=True),
            LogisticRegression(max_iter=1000, class_weight="balanced"),
        ),
        "Árbol de decisión": make_pipeline(
            preprocesador(),
            DecisionTreeClassifier(
                max_depth=5,
                min_samples_leaf=50,
                class_weight="balanced",
                random_state=SEMILLA,
            ),
        ),
    }

## 1. Clasificación binaria: V1 y V2

In [ ]:
escenarios = {
    "V1: benigno + SYN": [0, 1],
    "V2: benigno + SYN + UDP": [0, 1, 2],
}

filas_val, filas_test = [], []
binarios = {}

for escenario, clases in escenarios.items():
    subconjunto = train[train["clase"].isin(clases)]

    for nombre, modelo in modelos_base().items():
        modelo.fit(subconjunto[variables], subconjunto["etiqueta"])
        binarios[(escenario, nombre)] = modelo

        for filas, datos in [(filas_val, val), (filas_test, test)]:
            filas.append({
                "Escenario": escenario,
                "Modelo": nombre,
                **evaluar(modelo, datos[variables], datos["etiqueta"],
                          "binaria", clase_real=datos["clase"]),
            })

binaria_val = pd.DataFrame(filas_val).set_index(["Escenario", "Modelo"])
binaria_test = pd.DataFrame(filas_test).set_index(["Escenario", "Modelo"])

print("Validación")
display(binaria_val.round(4))
print("Prueba oficial")
display(binaria_test.round(4))

guardar_tabla(binaria_val, "metricas_validacion")
guardar_tabla(binaria_test, "metricas_prueba_oficial")

**Cómo leer la tabla:** en V1, el recall de UDP muestra si el modelo
reconoce una variante de DDoS que no vio en entrenamiento. Un recall
cercano a 0 indica la "ceguera ante vectores DDoS" descrita en el
informe. En V2, esa variante sí forma parte del entrenamiento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, escenario in zip(axes, escenarios):
    modelo = binarios[(escenario, "Árbol de decisión")]
    matriz_confusion(
        test["etiqueta"], modelo.predict(test[variables]),
        "binaria", f"Árbol — {escenario}", ax=ax,
    )
fig.suptitle("Prueba oficial: matriz de confusión binaria")
plt.tight_layout()
plt.show()

# Detección según la variante real de cada registro
for escenario in escenarios:
    modelo = binarios[(escenario, "Árbol de decisión")]
    tabla = pd.crosstab(
        test["clase"].map(CLASES),
        np.where(modelo.predict(test[variables]) == 1, "Predicho DDoS", "Predicho benigno"),
        rownames=["Clase real"],
        colnames=["Predicción"],
        normalize="index",
    )
    print(f"\n{escenario} — proporción por clase real (prueba oficial)")
    display(tabla.round(4))

## 2. Clasificación multiclase

In [ ]:
filas_val, filas_test = [], []
multiclase = {}

for nombre, modelo in modelos_base().items():
    modelo.fit(train[variables], train["clase"])
    multiclase[nombre] = modelo

    filas_val.append({"Modelo": nombre, **evaluar(
        modelo, val[variables], val["clase"], "multiclase")})
    filas_test.append({"Modelo": nombre, **evaluar(
        modelo, test[variables], test["clase"], "multiclase")})

multi_val = pd.DataFrame(filas_val).set_index("Modelo")
multi_test = pd.DataFrame(filas_test).set_index("Modelo")

print("Validación")
display(multi_val.round(4))
print("Prueba oficial")
display(multi_test.round(4))

guardar_tabla(multi_val, "metricas_multiclase_base_validacion")
guardar_tabla(multi_test, "metricas_multiclase_base_prueba_oficial")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, nombre in zip(axes, ["Regresión logística", "Árbol de decisión"]):
    matriz_confusion(
        test["clase"], multiclase[nombre].predict(test[variables]),
        "multiclase", nombre, ax=ax, normalizar="true",
    )
fig.suptitle("Prueba oficial: matriz de confusión multiclase (proporción por clase real)")
plt.tight_layout()
plt.show()

## 3. Variables utilizadas por el árbol multiclase

In [ ]:
arbol = multiclase["Árbol de decisión"]
importancias = pd.Series(
    arbol[-1].feature_importances_, index=variables
).sort_values(ascending=False)

display(importancias.head(10).rename("Importancia").to_frame().round(4))

importancias.head(10).sort_values().plot.barh(
    figsize=(8, 5), title="Árbol multiclase: variables más utilizadas"
)
plt.xlabel("Importancia dentro del modelo")
plt.tight_layout()
plt.show()

In [ ]:
CARPETA_MODELOS.mkdir(parents=True, exist_ok=True)
joblib.dump(binarios[("V2: benigno + SYN + UDP", "Árbol de decisión")],
            CARPETA_MODELOS / "arbol_binario.joblib")
joblib.dump(multiclase["Árbol de decisión"],
            CARPETA_MODELOS / "arbol_multiclase.joblib")
print("Modelos guardados en", CARPETA_MODELOS)

## Conclusiones de la línea base

Completar con los resultados obtenidos:

1. Diferencia entre la clase mayoritaria y los modelos entrenados,
   especialmente en F1 macro y tasa de falsos positivos.
2. Efecto de no incluir UDP en entrenamiento (V1 frente a V2).
3. Diferencia entre validación y prueba oficial (estabilidad dentro de IoMT).
4. Clases que se confunden en el caso multiclase.

Estos valores son la referencia que deben superar los modelos de
ensamble del notebook 04.